In [1]:
import os
import re
import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup

# --- CONFIGURATION ---
BASE_URL = "https://www.bolsadevalores.com.py/"  # Example portal URL
DOWNLOAD_DIR = "./bva_pdf_filings"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# ----------------------------------------------------------------------
# STEP 1: DOWNLOAD PDF FILINGS USING BEAUTIFULSOUP
# ----------------------------------------------------------------------


def fetch_and_download_pdfs(target_url: str) -> list[str]:
    """Scrapes a given URL for links to financial statement PDFs and saves them locally."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    try:
        response = requests.get(target_url, headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"Error connecting to {target_url}: {e}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")
    downloaded_files = []

    # Find all anchor tags pointing to PDF files containing key financial keywords
    for link in soup.find_all("a", href=True):
        href = link["href"]
        if href.endswith(".pdf") and any(
            kw in href.lower()
            for kw in ["balance", "estados", "financieros", "prospecto"]
        ):
            pdf_url = href if href.startswith("http") else BASE_URL + href
            filename = os.path.basename(pdf_url)
            local_path = os.path.join(DOWNLOAD_DIR, filename)

            # Download file
            print(f"Downloading: {filename}...")
            pdf_res = requests.get(pdf_url, headers=headers, timeout=30)
            with open(local_path, "wb") as f:
                f.write(pdf_res.content)

            downloaded_files.append(local_path)

    return downloaded_files


# ----------------------------------------------------------------------
# STEP 2: PARSE PDF BALANCE SHEETS USING PDFPLUMBER
# ----------------------------------------------------------------------


def parse_financial_pdf(pdf_path: str) -> pd.DataFrame:
    """Extracts balance sheet and income statement tables from a local PDF using pdfplumber."""
    extracted_records = []

    # Financial keywords to target in Paraguayan filings (Spanish/Guaraní context)
    target_concepts = [
        "ACTIVO CORRIENTE",
        "ACTIVO NO CORRIENTE",
        "TOTAL ACTIVO",
        "PASIVO CORRIENTE",
        "PASIVO NO CORRIENTE",
        "TOTAL PASIVO",
        "PATRIMONIO NETO",
        "VENTAS NETAS",
        "GANANCIA BRUTA",
        "RESULTADO DEL EJERCICIO",
        "EBITDA",
    ]

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            # 1. First attempt: Extract structured tables
            tables = page.extract_tables()

            for table in tables:
                for row in table:
                    # Clean up empty/None cells
                    clean_row = [
                        cell.strip().replace("\n", " ")
                        for cell in row
                        if cell is not None
                    ]

                    if not clean_row or len(clean_row) < 2:
                        continue

                    concept = clean_row[0]

                    # Match row with financial concepts
                    if any(
                        tc in concept.upper() for tc in target_concepts
                    ):
                        # Extract numerical values from the remaining columns
                        numbers = [
                            re.sub(r"[^\d.-]", "", val)
                            for val in clean_row[1:]
                            if re.search(r"\d", val)
                        ]

                        extracted_records.append(
                            {
                                "PDF_Source": os.path.basename(pdf_path),
                                "Page": page_num,
                                "Financial_Concept": concept,
                                "Values": numbers,
                            }
                        )

            # 2. Fallback: Parse plain text if tables are unformatted
            if not tables:
                text = page.extract_text()
                if text:
                    for line in text.split("\n"):
                        for tc in target_concepts:
                            if tc in line.upper():
                                # Extract string line and trailing numeric values
                                numbers = re.findall(
                                    r"[-+]?\d{1,3}(?:\.\d{3})*(?:,\d+)?", line
                                )
                                extracted_records.append(
                                    {
                                        "PDF_Source": os.path.basename(
                                            pdf_path
                                        ),
                                        "Page": page_num,
                                        "Financial_Concept": line.strip(),
                                        "Values": numbers,
                                    }
                                )

    # Convert extracted records into a clean pandas DataFrame
    df_raw = pd.DataFrame(extracted_records)
    return df_raw


# ----------------------------------------------------------------------
# STEP 3: PIPELINE EXECUTION & DATA NORMALIZATION
# ----------------------------------------------------------------------

if __name__ == "__main__":
    # 1. Target URL (Replace with active CNV / BVA Issuer URL)
    target_filings_url = "https://www.bolsadevalores.com.py/"

    # 2. Run Downloader
    print("--- Starting BeautifulSoup Web Scraper ---")
    pdf_files = fetch_and_download_pdfs(target_filings_url)

    # If testing without downloading, point to local files
    # pdf_files = ["./bva_pdf_filings/sample_balance.pdf"]

    all_data = []

    # 3. Run PDF Parser
    print("\n--- Starting PDFPlumber Financial Parser ---")
    for pdf in pdf_files:
        print(f"Parsing: {pdf}")
        df_parsed = parse_financial_pdf(pdf)
        if not df_parsed.empty:
            all_data.append(df_parsed)

    # 4. Consolidate Data
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)

        print("\n--- Consolidated Financial Data Output ---")
        print(final_df.head(10))

        # Save to CSV for the Feature Engineering pipeline
        final_df.to_csv("paraguay_parsed_financials.csv", index=False)
        print("\nSaved extracted features to 'paraguay_parsed_financials.csv'")
    else:
        print(
            "\nNo financial data could be extracted. Verify PDF structure or layout."
        )

--- Starting BeautifulSoup Web Scraper ---

--- Starting PDFPlumber Financial Parser ---

No financial data could be extracted. Verify PDF structure or layout.
